# ¿Cuántos parlamentarios necesitaría un gobierno mundial?

1. Se construye un **Índice de Prosperidad Relativa (IPR)**: la media de la posición de cada país en cinco rankings (estabilidad, democracia, desigualdad, esperanza de vida y años de escolarización).
2. Se estudia cómo crece el tamaño del parlamento con la población en los países más prósperos y en los países libres.
3. Se extrapola a la población mundial (7.800 millones).

**Datos necesarios** en `staging/mundial/` (ver [`data/README.md`](../data/README.md)): fragilidad del Estado, Freedom in the World, Gini, esperanza de vida y población (Banco Mundial), años de escolarización, sistemas electorales, constituciones y códigos de país COW. Además, `raw/mapas/countries.geojson`.

El top 10 del IPR se publicó en [`resultados/top_10_ipr.csv`](../resultados/top_10_ipr.csv).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent))  # permite importar el paquete politicai desde notebooks/
from politicai.config import RAW_DIR, STAGING_DIR
from politicai.limpieza import clean_country_names, clean_df_format, melt_banco_mundial, ultimo_valor_por_grupo

pd.set_option("display.max_columns", None)

MUNDIAL_DIR = STAGING_DIR / "mundial"
GEOJSON_PATH = RAW_DIR / "mapas" / "countries.geojson"


def resumen(df):
    print("países:", df["country"].nunique(), "| filas:", df.shape)
    return df.head()

## Carga de datos

In [ ]:
df_fragility = clean_df_format(pd.read_excel(MUNDIAL_DIR / "fragilidad.xls"))
df_freedomhouse = clean_df_format(pd.read_excel(MUNDIAL_DIR / "freedom_house.xlsx"))
df_gini = clean_df_format(pd.read_excel(MUNDIAL_DIR / "gini_mundial.xls"))
df_esperanzavida = clean_df_format(pd.read_excel(MUNDIAL_DIR / "esperanza_vida.xls"))
df_escolarizacion = clean_df_format(pd.read_csv(MUNDIAL_DIR / "años_escolarizacion_medios.csv"))
df_sistemaselectorales = clean_df_format(pd.read_excel(MUNDIAL_DIR / "sistemas_electorales.xls"))
df_poblacion = clean_df_format(pd.read_excel(MUNDIAL_DIR / "poblacion.xls"))
df_cow = clean_df_format(pd.read_csv(MUNDIAL_DIR / "cow_country_codes.csv"))

La tabla de constituciones (Comparative Constitutions Project) es muy grande, así que se reduce una vez a las columnas que se usan y se guarda en `constituciones_reducido.csv`:

In [ ]:
CONSTITUCIONES_REDUCIDO = MUNDIAL_DIR / "constituciones_reducido.csv"
if not CONSTITUCIONES_REDUCIDO.exists():
    columnas = ["cowcode", "country", "year", "doctit", "hoselect", "hosterm", "hosterml",
                "hogelect", "hogterm", "hogterml", "hogdiss", "housenum", "fedunit"]
    pd.read_excel(MUNDIAL_DIR / "constituciones.xlsx")[columnas].to_csv(CONSTITUCIONES_REDUCIDO, index=False)
df_constituciones = clean_df_format(pd.read_csv(CONSTITUCIONES_REDUCIDO))

## Tabla por país y año

Los indicadores del IPR se unen con *inner join*, así que solo quedan los países con todos los datos. Los indicadores descriptivos (sistema electoral, población, constitución) se añaden con *left join*.

### Fragilidad del Estado

In [ ]:
df_base = df_fragility[["scode", "country", "year", "sfi"]].rename(columns={"scode": "code", "sfi": "state_fragility_idx"})
df_base = clean_country_names(df_base, column="country")
resumen(df_base)

### Freedom in the World

In [ ]:
df_freedomhouse_idx = df_freedomhouse[["country_territory", "status", "total", "edition"]].rename(
    columns={"status": "free_not_free", "total": "democracy_idx", "edition": "year"}
)
df_freedomhouse_idx = clean_country_names(df_freedomhouse_idx, column="country_territory")
df_base = df_base.merge(df_freedomhouse_idx, on=["country", "year"], how="inner")
resumen(df_base)

### Gini

Muchos países no publican el Gini todos los años, así que se usa el último dato disponible de cada uno.

In [ ]:
df_gini_idx = melt_banco_mundial(df_gini, "gini").dropna(subset=["gini"])
df_gini_idx = clean_country_names(df_gini_idx, column="country_name")
df_gini_idx = ultimo_valor_por_grupo(df_gini_idx, "country").drop(columns=["year"])
df_base = df_base.merge(df_gini_idx, on="country", how="inner")
resumen(df_base)

### Esperanza de vida y años de escolarización

A partir de aquí, las tablas se cruzan por el código de país del Banco Mundial (`code_wb`).

In [ ]:
df_esperanzavida_idx = melt_banco_mundial(df_esperanzavida, "esperanza_vida").drop(columns=["country_name"])
df_base = df_base.merge(df_esperanzavida_idx, on=["code_wb", "year"], how="inner")
resumen(df_base)

In [ ]:
df_escolarizacion_idx = df_escolarizacion[["code", "year", "average_total_years_of_schooling_for_adult_population"]].rename(
    columns={"code": "code_wb", "average_total_years_of_schooling_for_adult_population": "anios_escolarizacion"}
)
df_escolarizacion_idx["year"] = df_escolarizacion_idx["year"].astype(int)
df_base = df_base.merge(df_escolarizacion_idx, on=["code_wb", "year"], how="inner")
resumen(df_base)

### Sistema electoral y población

In [ ]:
df_sistemaselectorales_idx = clean_country_names(df_sistemaselectorales, column="country")
df_sistemaselectorales_idx["year"] = df_sistemaselectorales_idx["year"].astype(int)
df_sistemaselectorales_idx = ultimo_valor_por_grupo(df_sistemaselectorales_idx, "country").drop(columns=["year"])
df_base = df_base.merge(df_sistemaselectorales_idx, on="country", how="left")

df_poblacion_idx = melt_banco_mundial(df_poblacion, "poblacion").drop(columns=["country_name"])
df_base = df_base.merge(df_poblacion_idx, on=["code_wb", "year"], how="left")
resumen(df_base)

### Constituciones

Se usan los códigos de país del proyecto Correlates of War (COW) para cruzar las constituciones con el resto. Algunos códigos se corrigen para que coincidan con los de la tabla de fragilidad.

In [ ]:
df_cow = df_cow.rename(columns={"ccode": "cowcode"})[["stateabb", "cowcode", "statenme"]].drop_duplicates()

df_constituciones_idx = df_constituciones.merge(df_cow, on="cowcode", how="inner").dropna(subset=["doctit"])
df_constituciones_idx = ultimo_valor_por_grupo(df_constituciones_idx, "cowcode")
df_constituciones_idx = df_constituciones_idx.drop(columns=["cowcode", "country", "year"]).rename(columns={"stateabb": "code"})

CODIGOS_COW_A_FRAGILIDAD = {"DRV": "VIE", "YAR": "YEM", "SSD": "SSU", "SUD": "SDN", "ETH": "ETI",
                            "ROM": "RUM", "DRC": "ZAI", "YUG": "SER", "CDI": "IVO"}
df_constituciones_idx["code"] = df_constituciones_idx["code"].replace(CODIGOS_COW_A_FRAGILIDAD)

df_base = df_base.merge(df_constituciones_idx, on="code", how="left")
resumen(df_base)

## Índice de Prosperidad Relativa (IPR)

Con el dato más reciente de cada país se ordena a los países en cada indicador (1 = mejor) y se hace la media de las cinco posiciones. Cuanto más bajo es el IPR, más próspero es el país.

In [ ]:
df_prosperidad = ultimo_valor_por_grupo(df_base, "country").copy()

RANKINGS = {
    # columna del ranking: (indicador, ¿un valor alto es mejor?)
    "rkg_stability": ("state_fragility_idx", False),
    "rkg_democracy": ("democracy_idx", True),
    "rkg_gini": ("gini", False),
    "rkg_life_exp": ("esperanza_vida", True),
    "rkg_school_yrs": ("anios_escolarizacion", True),
}
for ranking, (indicador, mas_es_mejor) in RANKINGS.items():
    df_prosperidad[ranking] = df_prosperidad[indicador].rank(ascending=not mas_es_mejor, method="min")

df_prosperidad["rkg_global"] = df_prosperidad[list(RANKINGS)].mean(axis=1)
df_prosperidad = df_prosperidad.sort_values("rkg_global")
df_prosperidad.head(10)

In [ ]:
fig_top10 = px.bar(
    df_prosperidad.head(10), x="statenme", y="rkg_global", hover_data=list(RANKINGS),
    labels={"statenme": "País", "rkg_global": "IPR"},
    title="Top 10 de países según el Índice de Prosperidad Relativa (IPR)",
)
fig_top10.show()

In [ ]:
with open(GEOJSON_PATH) as geofile:
    geojson = json.load(geofile)

fig_map_ipr = px.choropleth_map(
    data_frame=df_prosperidad.rename(columns={"rkg_global": "IPR"}), geojson=geojson,
    locations="code_wb", featureidkey="id", color="IPR", color_continuous_scale="RdYlBu_r",
    hover_data=["country"] + list(RANKINGS),
    map_style="carto-positron", zoom=0.5, center={"lat": 20, "lon": 0},
    title="Índice de Prosperidad Relativa (IPR) en el mundo",
)
fig_map_ipr.show()

## Tamaño del parlamento frente a población

In [ ]:
df_prosperidad["poblacion_millones"] = df_prosperidad["poblacion"] / 1e6
df_prosperidad["ratio_members_1M"] = df_prosperidad["legislative_size_voting_members"] / df_prosperidad["poblacion_millones"]

df_top = df_prosperidad.head(10)
df_libres = df_prosperidad[df_prosperidad["free_not_free"] == "F"].copy()
hover = ["country", "poblacion", "legislative_size_voting_members"]

In [ ]:
fig_ratio = px.scatter(
    df_top, x="poblacion_millones", y="legislative_size_voting_members", hover_data=hover,
    trendline="ols", trendline_color_override="red",
    title="Parlamentarios frente a población en top 10 de países según IPR",
)
fig_ratio.show()

In [ ]:
fig_ratio_libres = px.scatter(
    df_libres, x="poblacion_millones", y="ratio_members_1M", hover_data=hover,
    trendline="lowess", trendline_color_override="red",
    title="Ratio de parlamentarios por millón de habitantes frente a población",
)
fig_ratio_libres.show()

In [ ]:
fig_ratio_log = px.scatter(
    df_libres, x="poblacion_millones", y="ratio_members_1M", hover_data=hover, log_x=True, log_y=True,
    title="Ratio de parlamentarios por millón de habitantes frente a población (log)",
)
fig_ratio_log.show()

## Extrapolación a un gobierno mundial

Tres estimaciones para 7.800 millones de habitantes:

1. La recta de regresión del top 10 del IPR.
2. El ratio de parlamentarios por habitante de la India (0,407 por millón), una democracia con una población del orden de la mundial.
3. Una regresión de proceso gaussiano sobre los datos en escala logarítmica de los países libres, que recoge que el ratio por habitante baja a medida que crece la población.

Como referencia: la Unión Europea tiene 446 millones de habitantes y 705 eurodiputados; Estados Unidos, 328 millones y 435 miembros en la Cámara de Representantes.

In [ ]:
POBLACION_MUNDIAL_MILLONES = 7800
RATIO_INDIA = 0.4071239  # diputados por millón de habitantes

params = px.get_trendline_results(fig_ratio).px_fit_results.iloc[0].params
print("Regresión top 10 IPR:", int(params[0] + params[1] * POBLACION_MUNDIAL_MILLONES))
print("Ratio de la India:   ", int(RATIO_INDIA * POBLACION_MUNDIAL_MILLONES))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct

x = np.log(df_libres[["poblacion_millones"]])
y = np.log(df_libres["ratio_members_1M"])
modelo = GaussianProcessRegressor(kernel=DotProduct(), random_state=0).fit(x, y)

plt.scatter(x, y)
plt.plot(x, modelo.predict(x), c="red")
plt.xlabel("log(población, millones)")
plt.ylabel("log(parlamentarios por millón)")
plt.show()

x_mundo = np.log(pd.DataFrame({"poblacion_millones": [POBLACION_MUNDIAL_MILLONES]}))
ratio_mundo = float(np.exp(modelo.predict(x_mundo))[0])
print(f"Proceso gaussiano: {ratio_mundo:.3f} parlamentarios por millón -> {ratio_mundo * POBLACION_MUNDIAL_MILLONES:.0f} parlamentarios")

Resultados del análisis original: la regresión del top 10 da unas **52.411** personas (poco realista), el ratio de la India unas **3.175** y el proceso gaussiano unas **2.405**.